# Multi-Task Learning with Nash-MTL Loss Balancing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/multi_task/multi_task.ipynb)

This notebook demonstrates **multi-task learning** with Ludwig: training a single model to predict multiple outputs simultaneously, and using **loss balancing** to prevent one task from dominating training.

**Use case:** The [UCI Wine Quality dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) has a 0–10 quality score. We simultaneously predict:
- `quality_score` — the raw numerical score (regression)
- `quality_binary` — whether the wine is good (quality ≥ 7, binary classification)

These two tasks have different loss magnitudes. Without balancing, regression loss dominates and the classifier under-trains.

**Methods compared:**

| Method | Status | Description |
|---|---|---|
| `none` | Available | Static weighted sum — baseline |
| `famo` | Available | Fast Adaptive Multitask Optimization (Liu et al., NeurIPS 2023) |
| `uncertainty` | Available | Homoscedastic uncertainty weighting (Kendall et al., CVPR 2018) |
| `nash_mtl` | Requires PR #4092 | Nash bargaining solution (Navon et al., ICML 2022) |

In [ ]:
!pip install ludwig --quiet

In [ ]:
import logging
import os
import shutil
import warnings

import pandas as pd

logging.basicConfig(level=logging.WARNING)

## Dataset

We use the [UCI Wine Quality dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) (red wine, ~1 600 rows, 11 physicochemical input features).

Two output columns are derived from the original `quality` score:
- `quality_score`: the raw numerical score (0–10) — a regression target
- `quality_binary`: 1 if quality ≥ 7 ("good"), else 0 — a classification target

These tasks share the same inputs and are naturally correlated, making them a good multi-task setup.

In [ ]:
WINE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)

WINE_FEATURES = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide",
    "density", "pH", "sulphates", "alcohol",
]

print("Downloading wine quality dataset...")
df = pd.read_csv(WINE_URL, sep=";")
df.columns = [c.replace(" ", "_") for c in df.columns]

df["quality_score"] = df["quality"].astype(float)
df["quality_binary"] = (df["quality"] >= 7).astype(int)
df = df.drop(columns=["quality"])

print(f"Shape: {df.shape}")
print(f"Good wines (quality >= 7): {df['quality_binary'].mean():.1%}")
print(f"Quality score range: {df['quality_score'].min():.0f} – {df['quality_score'].max():.0f}")
df.head()

### Why does loss balancing matter here?

MSE for regression and binary cross-entropy for classification operate on different scales. Typically:
- Regression MSE: values in the range 0.5–2.0 (for a 0–10 quality score)
- Binary cross-entropy: values in the range 0.3–0.7

Without balancing, the regression task provides larger raw gradients and the model updates are biased toward minimising the regression loss at the expense of classification performance.

In [ ]:
from ludwig.api import LudwigModel

def input_features():
    return [
        {"name": feat, "type": "number", "preprocessing": {"normalization": "zscore"}}
        for feat in WINE_FEATURES
    ]

def make_config(loss_balancing: str) -> dict:
    return {
        "model_type": "ecd",
        "input_features": input_features(),
        "output_features": [
            {"name": "quality_score", "type": "number"},
            {"name": "quality_binary", "type": "binary"},
        ],
        "combiner": {
            "type": "concat",
            "num_fc_layers": 2,
            "output_size": 128,
            "dropout": 0.1,
        },
        "trainer": {
            "epochs": 30,
            "learning_rate": 0.001,
            "batch_size": 128,
            "loss_balancing": loss_balancing,
        },
    }

def train_and_evaluate(name: str, loss_balancing: str) -> dict | None:
    result_dir = f"./results/{name}"
    shutil.rmtree(result_dir, ignore_errors=True)
    print(f"\n{'='*50}")
    print(f"Training: {name}  (loss_balancing={loss_balancing})")
    print(f"{'='*50}")
    config = make_config(loss_balancing)
    model = LudwigModel(config=config, logging_level=logging.WARNING)
    train_stats, _, _ = model.train(
        dataset=df,
        experiment_name="multi_task",
        model_name=name,
        output_directory=result_dir,
    )
    # Extract final validation metrics
    vset = train_stats.get("validation", {})
    score_mae = _last(vset.get("quality_score", {}).get("mean_absolute_error", []))
    binary_auc = _last(vset.get("quality_binary", {}).get("roc_auc", []))
    print(f"  quality_score  MAE     : {score_mae:.4f}")
    print(f"  quality_binary ROC-AUC : {binary_auc:.4f}")
    return {"method": name, "score_mae": score_mae, "binary_roc_auc": binary_auc}

def _last(series):
    if not series:
        return float("nan")
    v = series[-1]
    if isinstance(v, (list, tuple)):
        v = v[-1]
    return float(v)

results = []

## Baseline: No Loss Balancing

The default `loss_balancing: none` computes the total loss as a static weighted sum:

```
L_total = w_score * L_score + w_binary * L_binary
```

where weights are set from the config (defaulting to 1.0 each). The regression task typically has a larger loss value, so it tends to dominate gradient updates.

Config snippet:
```yaml
trainer:
  loss_balancing: none
```

In [ ]:
result = train_and_evaluate("none", "none")
results.append(result)

## FAMO Balancing

> **Available now** in the current Ludwig release.

**FAMO** (Fast Adaptive Multitask Optimization, Liu et al., NeurIPS 2023) maintains an exponential moving average of each task's loss and updates task weights at every step to equalise the rate of loss decrease across tasks.

Key properties:
- No gradient computation overhead (unlike gradient-based methods)
- Converges quickly; good default choice for most multi-task problems
- One hyperparameter: `loss_balancing_lr` (EMA learning rate, default 0.025)

Config snippet:
```yaml
trainer:
  loss_balancing: famo
```

In [ ]:
result = train_and_evaluate("famo", "famo")
results.append(result)

## Uncertainty Weighting

> **Available now** in the current Ludwig release.

**Uncertainty weighting** (Kendall et al., CVPR 2018) treats each task's weight as a learned parameter representing homoscedastic (task-level) uncertainty. Tasks with higher uncertainty receive lower weight automatically.

Key properties:
- Learns a single scalar per task — minimal parameter overhead
- Principled Bayesian interpretation
- Works best when tasks have stable, intrinsically different noise levels

Config snippet:
```yaml
trainer:
  loss_balancing: uncertainty
```

In [ ]:
result = train_and_evaluate("uncertainty", "uncertainty")
results.append(result)

## Nash-MTL Balancing

> **Requires PR #4092** (`future-capabilities` branch). Not yet available in the main Ludwig release.
> 
> To try it now:
> ```bash
> pip install git+https://github.com/ludwig-ai/ludwig@future-capabilities
> ```

**Nash-MTL** (Navon et al., ICML 2022) finds the [Nash bargaining solution](https://en.wikipedia.org/wiki/Nash_bargaining_solution) for task weight allocation. In game-theoretic terms, it selects a Pareto-optimal point where no task can improve its loss without worsening another task's loss.

Concretely, Nash-MTL solves a quadratic programme at each step to find weights `alpha` such that:

```
alpha = argmax  sum_i log(alpha_i * loss_i)   subject to  sum alpha_i = 1, alpha_i >= 0
```

This is equivalent to weights being inversely proportional to task losses — tasks with larger losses receive proportionally higher weight, equalising marginal contributions.

Key properties:
- Theoretically principled (game theory, not heuristic)
- Most effective when tasks genuinely conflict (gradients point in different directions)
- Slightly higher per-step overhead than FAMO (solves a small QP per step)

Config snippet:
```yaml
trainer:
  loss_balancing: nash_mtl  # Requires PR #4092 / Ludwig >= 0.14
```

Reference: Navon et al., [Multi-Task Learning as a Bargaining Game](https://arxiv.org/abs/2202.01017), ICML 2022.

In [ ]:
# Nash-MTL is attempted; a warning is printed if PR #4092 is not yet available.
try:
    result = train_and_evaluate("nash_mtl", "nash_mtl")
    results.append(result)
except Exception as exc:
    warnings.warn(
        f"nash_mtl is not available in this Ludwig version ({exc}). "
        "Install from 'future-capabilities' branch to enable it."
    )
    results.append({"method": "nash_mtl", "score_mae": float("nan"), "binary_roc_auc": float("nan")})

## Comparison Table

Side-by-side comparison of all four methods on the validation split.

In [ ]:
import pandas as pd

comparison = pd.DataFrame(results).set_index("method")
comparison.columns = ["Score MAE (lower better)", "Binary ROC-AUC (higher better)"]
print(comparison.to_string())
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

methods = comparison.index.tolist()
mae_vals = comparison["Score MAE (lower better)"].tolist()
auc_vals = comparison["Binary ROC-AUC (higher better)"].tolist()

colors = ["tab:gray", "tab:blue", "tab:orange", "tab:green"]

axes[0].bar(methods, mae_vals, color=colors[:len(methods)])
axes[0].set_title("Quality Score — MAE (lower is better)")
axes[0].set_ylabel("MAE")
axes[0].set_ylim(0, max(mae_vals) * 1.2)

axes[1].bar(methods, auc_vals, color=colors[:len(methods)])
axes[1].set_title("Quality Binary — ROC-AUC (higher is better)")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150)
plt.show()

## When to Use Nash-MTL

Use **Nash-MTL** when:

- You have two or more output features with significantly different loss scales
- Your tasks conflict (improving one task hurts another), as measured by negative gradient cosine similarity
- You want the most principled, theoretically grounded balancing approach
- You can tolerate a small additional per-step compute cost (solving a small QP)

Use **FAMO** when:
- You want a strong default with no tuning required
- Training speed is a priority (FAMO has lower overhead than Nash-MTL)

Use **uncertainty weighting** when:
- Tasks have stable, intrinsically different noise levels
- You want a simple, interpretable approach with a Bayesian justification

Use **none** (baseline) when:
- You have a single output feature
- Your tasks have similar loss scales and do not conflict
- You have manually tuned `loss_weight` values on each output feature

### Further Reading

- Navon et al., [Multi-Task Learning as a Bargaining Game](https://arxiv.org/abs/2202.01017), ICML 2022
- Liu et al., [FAMO: Fast Adaptive Multitask Optimization](https://arxiv.org/abs/2306.03792), NeurIPS 2023
- Kendall et al., [Multi-Task Learning Using Uncertainty to Weigh Losses](https://arxiv.org/abs/1705.07115), CVPR 2018
- Chen et al., [GradNorm: Gradient Normalization for Adaptive Loss Balancing](https://arxiv.org/abs/1711.02257), ICML 2018